In [1]:
import os
from pyspark.sql import SparkSession

BUCKET = os.environ["KUBERNETES_NAMESPACE"]
ICEBERG_PACKAGE = "org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.11.0"

spark = (
    SparkSession.builder
    .appName("lab-iceberg")
    .master("local[*]")
    .config("spark.hadoop.fs.s3a.endpoint", f"https://{os.environ['AWS_S3_ENDPOINT']}")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.session.token", os.environ.get("AWS_SESSION_TOKEN", ""))
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.jars.packages", ICEBERG_PACKAGE)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lab", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lab.type", "hadoop")
    .config("spark.sql.catalog.lab.warehouse", f"s3a://{BUCKET}/warehouse")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(spark.version, BUCKET)

:: loading settings :: url = jar:file:/usr/local/lib/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/onyxia/.ivy2.5.2/cache
The jars for the packages stored in: /home/onyxia/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-13ccc75e-1379-4376-a0cf-497ba679da52;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 in central
:: resolution report :: resolve 225ms :: artifacts dl 15ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-4.1_2.13;1.11.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |

4.1.1 user-k-sison-ece


In [2]:
spark.sql("SHOW NAMESPACES IN lab").show()
spark.sql("SHOW TABLES IN lab.silver").show()

+---------+
|namespace|
+---------+
|   silver|
+---------+

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|   silver|    users|      false|
|   silver|   orders|      false|
+---------+---------+-----------+



In [3]:
spark.sql("""
    SELECT count(*) AS orders, sum(quantity) AS quantity
    FROM lab.silver.orders
""").show()
spark.table("lab.silver.orders").printSchema()

+------+--------+
|orders|quantity|
+------+--------+
|  2829|    8615|
+------+--------+

root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ordered_at: timestamp (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- product: string (nullable = true)



In [4]:
spark.sql("""
    SELECT *
    FROM lab.silver.orders.snapshots
""").show(truncate=False)

+-----------------------+-------------------+---------+---------+----------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id|operation|manifest_list                                                                                                               |summary                                           

In [ ]:
spark.sql("""
    SELECT file_path, file_format, record_count, file_size_in_bytes
    FROM lab.silver.orders.files
""").show(truncate=False)

In [2]:
spark.sql("""
    SELECT operation, summary
    FROM lab.silver.orders.snapshots
""").show(truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 operation | overwrite                                                                                                                                                                                                                                                                                                                                                                                                           

In [3]:
from pyspark.sql import functions as F

stg_orders = spark.table("lab.silver.orders")
stg_orders.show(5)

+--------------------+--------------------+--------------------+--------+---------+
|            order_id|             user_id|          ordered_at|quantity|  product|
+--------------------+--------------------+--------------------+--------+---------+
|00060d5a-02b3-494...|f53d4330-cdda-44b...|2020-01-28 12:31:...|       2|   cookie|
|0011be30-e268-492...|5f733a3e-5d28-4aa...|2020-04-18 22:51:...|       5|    bread|
|002394e2-2ad9-468...|90164161-cfa7-41c...|2020-02-24 01:03:...|       4|   cookie|
|0041f5d4-9549-463...|aab612c9-415d-474...|2020-03-17 18:29:...|       2|croissant|
|00684d9c-d2bf-4c3...|97fa7f04-8363-400...|2020-03-30 13:36:...|       4|    bread|
+--------------------+--------------------+--------------------+--------+---------+
only showing top 5 rows


In [4]:
new_orders = stg_orders.limit(3).withColumn("order_id", F.expr("uuid()"))
new_orders.writeTo("lab.silver.orders").append()

In [5]:
spark.sql("""
    SELECT snapshot_id, parent_id, operation, committed_at
    FROM lab.silver.orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-------------------+-------------------+---------+-----------------------+
|snapshot_id        |parent_id          |operation|committed_at           |
+-------------------+-------------------+---------+-----------------------+
|4219974134046780544|NULL               |overwrite|2026-10-28 15:48:20.462|
|7719747662070971571|4219974134046780544|append   |2026-11-03 19:54:57    |
+-------------------+-------------------+---------+-----------------------+



In [6]:
spark.sql("SELECT count(*) AS orders FROM lab.silver.orders").show()

+------+
|orders|
+------+
|  2832|
+------+



In [7]:
snapshots = spark.sql("""
    SELECT snapshot_id
    FROM lab.silver.orders.snapshots
    ORDER BY committed_at
""").collect()

snapshots

[Row(snapshot_id=4219974134046780544), Row(snapshot_id=7719747662070971571)]

In [8]:
first_snapshot = snapshots[0]["snapshot_id"]

spark.sql(f"""
    SELECT count(*) AS orders
    FROM lab.silver.orders
    VERSION AS OF {first_snapshot}
""").show()

+------+
|orders|
+------+
|  2829|
+------+



In [9]:
spark.sql("""
    SELECT count(*) AS orders
    FROM lab.silver.orders
""").show()

+------+
|orders|
+------+
|  2832|
+------+



In [10]:
spark.sql("""
    ALTER TABLE lab.silver.orders
    ADD COLUMN source STRING
""")

DataFrame[]

In [11]:
spark.sql("DESCRIBE lab.silver.orders").show()

+----------+---------+-------+
|  col_name|data_type|comment|
+----------+---------+-------+
|  order_id|   string|   NULL|
|   user_id|   string|   NULL|
|ordered_at|timestamp|   NULL|
|  quantity|      int|   NULL|
|   product|   string|   NULL|
|    source|   string|   NULL|
+----------+---------+-------+



In [12]:
spark.sql("""
    INSERT INTO lab.silver.orders
    SELECT order_id, user_id, ordered_at, quantity, product, 'spark'
    FROM lab.silver.orders
    LIMIT 1
""")

DataFrame[]

In [13]:
spark.sql("""
    SELECT source, count(*) AS orders
    FROM lab.silver.orders
    GROUP BY source
""").show()

spark.sql("""
    SELECT snapshot_id, parent_id, operation, committed_at
    FROM lab.silver.orders.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+------+------+
|source|orders|
+------+------+
|  NULL|  2832|
| spark|     1|
+------+------+

+-------------------+-------------------+---------+-----------------------+
|snapshot_id        |parent_id          |operation|committed_at           |
+-------------------+-------------------+---------+-----------------------+
|4219974134046780544|NULL               |overwrite|2026-10-28 15:48:20.462|
|7719747662070971571|4219974134046780544|append   |2026-11-03 19:54:57    |
|1220184473976417285|7719747662070971571|append   |2026-11-03 19:59:47.577|
+-------------------+-------------------+---------+-----------------------+



In [14]:
spark.sql("SELECT count(*) AS users FROM lab.silver.users").show()
spark.table("lab.silver.users").printSchema()
spark.sql("""
    SELECT snapshot_id, parent_id, operation, committed_at
    FROM lab.silver.users.snapshots ORDER BY committed_at
""").show(truncate=False)

+-----+
|users|
+-----+
|   50|
+-----+

root
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sex: string (nullable = true)
 |-- email: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)

+-------------------+---------+---------+-----------------------+
|snapshot_id        |parent_id|operation|committed_at           |
+-------------------+---------+---------+-----------------------+
|1533708320527085805|NULL     |overwrite|2026-10-28 15:48:18.152|
+-------------------+---------+---------+-----------------------+



In [15]:
from pyspark.sql import functions as F

users = spark.table("lab.silver.users")
new_users = users.limit(5).withColumn("user_id", F.expr("uuid()"))
new_users.writeTo("lab.silver.users").append()

spark.sql("SELECT count(*) AS users FROM lab.silver.users").show()
spark.sql("""
    SELECT snapshot_id, parent_id, operation, committed_at
    FROM lab.silver.users.snapshots ORDER BY committed_at
""").show(truncate=False)

+-----+
|users|
+-----+
|   55|
+-----+

+-------------------+-------------------+---------+-----------------------+
|snapshot_id        |parent_id          |operation|committed_at           |
+-------------------+-------------------+---------+-----------------------+
|1533708320527085805|NULL               |overwrite|2026-10-28 15:48:18.152|
|7345446893973528675|1533708320527085805|append   |2026-11-03 20:05:57.42 |
+-------------------+-------------------+---------+-----------------------+



In [16]:
users_snapshots = spark.sql("""
    SELECT snapshot_id FROM lab.silver.users.snapshots ORDER BY committed_at
""").collect()
first_users_snapshot = users_snapshots[0]["snapshot_id"]

spark.sql(f"""
    SELECT count(*) AS users FROM lab.silver.users VERSION AS OF {first_users_snapshot}
""").show()
spark.sql("SELECT count(*) AS users FROM lab.silver.users").show()

+-----+
|users|
+-----+
|   50|
+-----+

+-----+
|users|
+-----+
|   55|
+-----+



In [17]:
spark.sql("ALTER TABLE lab.silver.users ADD COLUMN country STRING")

batch = (
    spark.table("lab.silver.users").limit(2)
    .withColumn("user_id", F.expr("uuid()"))
    .withColumn("country", F.lit("France"))
)
batch.writeTo("lab.silver.users").append()

spark.sql("""
    SELECT country, count(*) AS users FROM lab.silver.users GROUP BY country
""").show()

+-------+-----+
|country|users|
+-------+-----+
| France|    2|
|   NULL|   55|
+-------+-----+



In [18]:
spark.stop()